# 06 - Evaluation

Up to now we judged the assistant by trying a handful of questions and looking at the answers. That gives a feeling, not a measurement. An **evaluation** runs the same fixed set of questions every time and turns the results into numbers, so we can (1) say how good the system is, (2) compare two designs fairly (notebook 02 did that for chunking), and (3) notice when a change makes things worse.

This notebook explains the gold set, each metric, how to run the evaluation, and the results we have so far. **Status:** only the retrieval part has been run. Answer quality needs a Gemini API key, and the gold set still needs Pooja's review, so those numbers are pending.

In [1]:
import os
import warnings

# Silence noise before anything is imported (deprecation notices, HF logs).
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_VERBOSITY"] = "error"

import json  # noqa: E402
import sys  # noqa: E402
import textwrap  # noqa: E402
from pathlib import Path  # noqa: E402

import pandas as pd  # noqa: E402

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project root so data/ and eval/ paths work
sys.path.insert(0, str(Path.cwd()))  # so that `import eval...` works

from eval.metrics import hit_at_k, percentile  # noqa: E402
from eval.run_eval import label, load_gold  # noqa: E402

gold = load_gold()  # also validates the format
df = pd.DataFrame(gold)
print(len(gold), "questions")
print(df.groupby(["language", "out_of_scope"]).size().to_string())

40 questions
language  out_of_scope
de        False           17
          True             3
en        False           18
          True             2


## The gold set (`eval/gold.jsonl`)

A *gold set* is a list of questions together with the answer we expect. Here the "answer" is not text but the **provision(s)** that answer the question. One line per question:

- `id`, `question`, `language` (`de` / `en`)
- `expected`: a list of provisions (`regulation`, `type`, `number`, `paragraph`). `paragraph: null` means "any paragraph of this article is fine". With several entries, finding *any* of them counts (useful for cross-law questions).
- `out_of_scope`: `true` for questions the assistant should refuse (tax, minimum wage, ...); they have `expected: []`.
- `note`: why this provision answers the question, so a reviewer can check it.

40 questions: 20 German, 20 English; 15 AI Act, 15 GDPR, 5 cross-law, 5 out of scope. Every expected provision was checked against the actual chunk text (the consolidated AI Act has amendments: for example the bias-detection rule moved from Art. 10(5) to a new Art. 4a).

In [2]:
for g in gold[:1] + gold[31:32] + gold[35:36]:
    print(json.dumps(g, ensure_ascii=False))

{"id": "q01", "question": "Which uses of AI are completely banned in the EU, for example social scoring?", "language": "en", "expected": [{"regulation": "AI_ACT", "type": "article", "number": "5", "paragraph": "1"}], "out_of_scope": false, "note": "Art. 5(1) lists the prohibited AI practices; point (c) is social scoring."}
{"id": "q32", "question": "Darf ein Anbieter von Hochrisiko-KI sensible Daten wie die ethnische Herkunft verarbeiten, um Verzerrungen zu erkennen?", "language": "de", "expected": [{"regulation": "AI_ACT", "type": "article", "number": "4a", "paragraph": "1"}], "out_of_scope": false, "note": "AI Act Art. 4a(1) (consolidated text): special categories for bias detection under strict conditions. In the original 2024 text this was Art. 10(5)."}
{"id": "q36", "question": "What is the corporate income tax rate in Germany?", "language": "en", "expected": [], "out_of_scope": true, "note": "Tax law, not in either regulation."}


## The metrics, in plain words

**Retrieval** (no LLM needed, free):

- **hit@k (article):** for how many in-scope questions is at least one expected article among the top *k* = 6 retrieved provisions? A retrieved Art. 6(3) counts for an expected Art. 6(2). If retrieval misses, the LLM cannot answer from the right text, so this is the ceiling for everything after it.
- **hit@k (paragraph):** the same, but the paragraph must match too. Only questions whose expectation names a paragraph count (n = 28).
- **retrieval latency p50 / p95:** p50 (the median) is the time half of the questions are faster than; p95 is the time 95% are faster than, so it shows the slow cases that an average would hide.

**Answers** (need the LLM):

- **citation correctness:** does the answer cite at least one expected article?
- **refusal accuracy:** does the assistant refuse exactly when it should (out-of-scope → refuse, in-scope → answer)? Counted over all 40 questions.
- **faithfulness (1-5):** is every claim in the answer supported by the passages it was given? Graded by a second LLM call (an "LLM judge") with a fixed rubric, returning JSON.
- **relevance (1-5):** does the answer address the question that was asked? Same judge.
- **answer latency, avg tokens:** speed and cost per answer.

Two of these on toy data:

In [3]:
expected = [{"regulation": "AI_ACT", "type": "article", "number": "6", "paragraph": "2"}]
retrieved = [("GDPR", "article", "6", "1"), ("AI_ACT", "article", "6", "3")]
print("article level:  ", hit_at_k(retrieved, expected, "article"))  # Art. 6(3) is in Art. 6
print("paragraph level:", hit_at_k(retrieved, expected, "paragraph"))  # but it is not 6(2)
latencies = [20, 22, 25, 26, 27, 30, 400]  # one slow outlier
print(
    "mean",
    round(sum(latencies) / len(latencies)),
    "| p50",
    percentile(latencies, 50),
    "| p95",
    round(percentile(latencies, 95)),
)

article level:   True
paragraph level: False
mean 79 | p50 26.0 | p95 289


The mean (79 ms) describes none of the requests: six are fast, one is very slow. p50 and p95 tell that story.

## How to run it

```
# retrieval only (free, ~20 s; the naive run builds data/chroma_naive once, ~1.5 min)
uv run python eval/run_eval.py --retrieval-only --chunking structured
uv run python eval/run_eval.py --retrieval-only --chunking naive

# once GOOGLE_API_KEY is in .env:
uv run python eval/run_eval.py --limit 3 --skip-judge   # smoke test: 3 answers, no judge
uv run python eval/run_eval.py --skip-judge             # all answers
uv run python eval/run_eval.py                          # answers + judge
```

The free Gemini tier allows only a few requests per minute, so the script waits 6 seconds between LLM calls (`--throttle` changes that) and saves every answer and verdict in `data/cache/eval/<chunking>/<id>.json`. If the quota runs out, it stops calling the LLM, keeps what it has, and the next run continues where it stopped; nothing is asked twice. If you edit a question in the gold set, only that question is asked again. Results go to `eval/results.md` and the per-question tables `eval/results_per_question.md` / `.csv`.

## Results so far (retrieval only)

In [4]:
runs = {
    name: json.loads(Path(f"eval/runs/{name}.json").read_text()) for name in ("structured", "naive")
}


def pct(m):
    return "pending" if m is None else f"{m['value']:.1%} (n={m['n']})"


def ms(m):
    return "pending" if m is None else f"{m['p50']:.0f} / {m['p95']:.0f} ms"


pd.DataFrame(
    {
        name: {
            "hit@k article": pct(r["summary"]["hit_article"]),
            "hit@k paragraph": pct(r["summary"]["hit_paragraph"]),
            "retrieval p50 / p95": ms(r["summary"]["retrieval_ms"]),
            "citation correctness": pct(r["summary"]["citation_correct"]),
            "refusal accuracy": pct(r["summary"]["refusal_accuracy"]),
        }
        for name, r in runs.items()
    }
)

,structured,naive
hit@k article,82.9% (n=35),54.3% (n=35)
hit@k paragraph,64.3% (n=28),32.1% (n=28)
retrieval p50 / p95,26 / 30 ms,23 / 26 ms
citation correctness,pending,pending
refusal accuracy,pending,pending


## Looking at the misses

A number alone does not say *why* the system fails. These are the 6 in-scope questions where structured retrieval did not find the expected article:

In [5]:
by_id = {g["id"]: g for g in gold}
misses = [r for r in runs["structured"]["records"] if r["hit_article"] is False]
pd.set_option("display.max_colwidth", 70)
pd.DataFrame(
    {
        "id": r["id"],
        "question": textwrap.shorten(by_id[r["id"]]["question"], 55),
        "expected": "; ".join(label(e) for e in r["expected"]),
        "top 3 retrieved": ", ".join(r["retrieved"][:3]),
    }
    for r in misses
)

,id,question,expected,top 3 retrieved
0,q02,"Darf ein Arbeitgeber KI einsetzen, um die [...]",AI Act Art. 5(1),"AI Act Recital 44, AI Act Recital 18, AI Act Art. 3(39)"
1,q16,Auf welcher Rechtsgrundlage darf ein Unternehmen [...],GDPR Art. 6(1),"GDPR Recital 48, GDPR Recital 47, GDPR Recital 69"
2,q21,Which kinds of personal data count as especially [...],GDPR Art. 9(1),"GDPR Recital 10, GDPR Recital 51, AI Act Recital 54"
3,q22,Welche Informationen muss ein Unternehmen mir [...],GDPR Art. 13,"GDPR Recital 48, GDPR Recital 70, GDPR Recital 110"
4,q30,"As a developer, do I have to build privacy [...]",GDPR Art. 25(1),"GDPR Recital 14, AI Act Art. 2(7), GDPR Recital 7"
5,q33,Does the AI Act replace the GDPR when an AI [...],AI Act Art. 2(7),"AI Act Art. 59(1), AI Act Art. 59(3), AI Act Recital 10"


A clear pattern: in 5 of the 6 misses the top results are **recitals**. Recitals explain the law in plain, question-like language ("Personal data which are, by their nature, particularly sensitive ... merit specific protection"), so their embeddings sit close to natural questions. The operative article uses formal language and loses. Two examples, the expected article next to what came first:

In [6]:
from assistant.parse import read_chunks  # noqa: E402

chunks = {c.id: c for c in read_chunks(Path("data/chunks.jsonl"))}
for qid, expected_id, top_id in [
    ("q21", "GDPR-en-article-9-1", "GDPR-en-recital-51"),
    ("q33", "AI_ACT-en-article-2-7", "AI_ACT-en-article-59-1"),
]:
    print(qid, by_id[qid]["question"])
    print("  expected:", textwrap.shorten(chunks[expected_id].text, 170))
    print("  retrieved:", textwrap.shorten(chunks[top_id].text, 170))
    print()

q21 Which kinds of personal data count as especially sensitive under the GDPR?
  expected: Article 9 – Processing of special categories of personal data (1) Processing of personal data revealing racial or ethnic origin, political opinions, religious or [...]
  retrieved: Recital 51 Personal data which are, by their nature, particularly sensitive in relation to fundamental rights and freedoms merit specific protection as the context [...]

q33 Does the AI Act replace the GDPR when an AI system processes personal data?
  expected: Article 2 – Scope (7) Union law on the protection of personal data, privacy and the confidentiality of communications applies to personal data processed in [...]
  retrieved: Article 59 – Further processing of personal data for developing certain AI systems in the public interest in the AI regulatory sandbox (1) In the AI regulatory [...]



- **q21** (sensitive data): Recital 51 lists the same categories as Art. 9(1). An answer built from it would be correct in content, but it would cite a recital instead of the binding article. Our gold set counts that as a miss; you could argue it should count as half right.
- **q33** (does the AI Act replace the GDPR?): the words "personal data" and "AI" pull in Art. 59 (re-using personal data in sandboxes), which shares the vocabulary but not the meaning. AI Act Recital 10, also retrieved, does say that data protection law stays untouched, so here too the right *idea* came back under a recital.
- **q16** (German, "Auf welcher Rechtsgrundlage ...") returned only recitals (GDPR Recitals 48 and 47, both about legitimate interests, first) instead of Art. 6(1), the list of legal bases.

We did **not** change the system to fix these, on purpose: tuning `top_k` or the weights on the same 35 questions we report would make the numbers look better without telling us whether the system got better (that is "overfitting the test"). Ideas for later, to decide together: give articles a small boost over recitals, or list the recital as an acceptable alternative in the gold set (only where the recital really answers the question).

## What these numbers prove, and what not

- They are measured, reproducible numbers on 35 in-scope questions, not impressions. Structured chunking beat naive chunking on every question where they differed.
- 35 is small: one question moves hit@k by about 3 points. A 95% confidence interval (Wilson) for 29 of 35 is roughly 67-92%, for naive's 19 of 35 roughly 38-70%. The *comparison* is firmer than the single numbers: on the 10 questions where the two disagreed, structured won all 10 (a sign test gives p ≈ 0.002), so on this question set the gap is very unlikely to be luck.
- We wrote the questions ourselves, knowing the system. They may be easier or more "article-shaped" than real users' questions, which flatters structured chunking.
- Retrieval hits are a ceiling, not an answer score: the LLM may still answer badly from the right passage, or well from a recital. The pending answer metrics measure that.
- The gold set was reviewed by the author on 2026-11-23 (no changes); n = 35 and self-written questions still make the numbers indicative.